# 🌿 Plant Disease Detector v2: 14 crops, 38 conditions

Version 2 uses the **full PlantVillage dataset** (about 54,000 colour photos) covering **14 crops**:
apple, blueberry, cherry, corn, grape, orange, peach, bell pepper, potato, raspberry, soybean, squash,
strawberry and tomato. It recognises **38 leaf conditions** (healthy + diseases).

The model uses **transfer learning**: MobileNetV2, already trained on millions of everyday images, is reused
as a feature extractor, and only a small new "head" is trained for our 38 classes, then the top layers are
fine-tuned. This is much more accurate than training a small CNN from scratch.

**Before you start:** Runtime → Change runtime type → **T4 GPU**, and make sure the Colab secret
`KAGGLE_API_TOKEN` exists (🔑 icon, Notebook access on).

Pipeline: Kaggle → download (colour photos only) → set aside 5 test photos per class → train/val split →
train head → fine-tune → save → test on the held-out photos.

## 1. Kaggle setup

In [ ]:
!pip -q install -U "kaggle>=1.8"
import os, shutil
from google.colab import files, userdata

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
!kaggle datasets files abdallahalidev/plantvillage-dataset --page-size 1 > /dev/null && echo "Kaggle sign-in OK"

## 2. Download the dataset (colour photos only)
The zip also holds grayscale and segmented copies; only the `color/` folder is extracted.

In [ ]:
import glob, pathlib

%cd /content
if not glob.glob('/content/pv/**/color', recursive=True):
    !kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content
    !unzip -q -o /content/plantvillage-dataset.zip '*color/*' -d /content/pv
    os.remove('/content/plantvillage-dataset.zip')

DATA_DIR = pathlib.Path(sorted(glob.glob('/content/pv/**/color', recursive=True))[0])
classes = sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir())
crops = sorted({c.split('___')[0] for c in classes})
n_images = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
print(f'{len(classes)} classes, {len(crops)} crops, {n_images:,} images in {DATA_DIR}')
print('Crops:', ', '.join(crops))

## 3. Set aside the test photos
**5 photos per class** (38 × 5 = 190) are **moved** out of the dataset before training, with a fixed seed so
it's the same set every run. The model never sees them during training or validation.

In [ ]:
import json, random
from collections import Counter

TEST_DIR = pathlib.Path('/content/test_samples_v2')
PER_CLASS = 5
IMG_EXTS = {'.jpg', '.jpeg', '.png'}
manifest_path = TEST_DIR / 'manifest.json'

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    for item in manifest:                      # in case the dataset was re-extracted
        (DATA_DIR / item['file']).unlink(missing_ok=True)
    print(f'Using the {len(manifest)} test photos already set aside.')
else:
    rng = random.Random(42)
    manifest = []
    for c in classes:
        imgs = sorted(p for p in (DATA_DIR / c).iterdir() if p.suffix.lower() in IMG_EXTS)
        for p in rng.sample(imgs, PER_CLASS):
            dest = TEST_DIR / c / p.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(p), dest)
            manifest.append({'file': f'{c}/{p.name}', 'label': c})
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f'Moved {len(manifest)} test photos to {TEST_DIR}')

print(Counter(m['label'].split('___')[0] for m in manifest))

## 4. Train / validation split (80 / 20)

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU') or 'none (training will be slow)')

IMG_SIZE   = (160, 160)
BATCH_SIZE = 64
SEED       = 123

train_data, val_data = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='both', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
)
class_names = train_data.class_names
num_classes = len(class_names)
print(num_classes, 'classes')

AUTOTUNE = tf.data.AUTOTUNE
train_data = train_data.prefetch(AUTOTUNE)
val_data   = val_data.prefetch(AUTOTUNE)

## 5. Model: MobileNetV2 transfer learning
- Augmentation (flip, rotation, zoom) only runs during training
- `Rescaling` **inside** the model, so the saved model takes raw 0–255 photos (the website sends raw pixels)
- MobileNetV2 base is **frozen** first; only the new classifier head learns

In [ ]:
from tensorflow.keras import layers

base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = layers.RandomFlip('horizontal')(inputs)
x = layers.RandomRotation(0.1)(x)
x = layers.RandomZoom(0.1)(x)
x = layers.Rescaling(1./127.5, offset=-1)(x)        # MobileNetV2 expects pixels in [-1, 1]
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary(show_trainable=True)

## 6. Train the head (stage 1)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
history1 = model.fit(train_data, validation_data=val_data, epochs=6, callbacks=[early_stop])

## 7. Fine-tune the top of MobileNetV2 (stage 2)
Unfreeze the last 30 layers and train them gently (very low learning rate) so the base adapts to leaves.

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop2 = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
history2 = model.fit(train_data, validation_data=val_data, epochs=6, callbacks=[early_stop2])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

h = {k: history1.history[k] + history2.history[k] for k in history1.history}
epochs = list(range(1, len(h['accuracy']) + 1))   # epochs labelled from 1
switch = len(history1.history['accuracy']) + 0.5  # where fine-tuning starts

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, h['accuracy'], label='train')
ax1.plot(epochs, h['val_accuracy'], label='validation')
ax1.set_title('Accuracy')
ax2.plot(epochs, h['loss'], label='train')
ax2.plot(epochs, h['val_loss'], label='validation')
ax2.set_title('Loss')
for ax in (ax1, ax2):
    ax.axvline(switch, ls='--', color='gray', label='fine-tuning starts')
    ax.set_xticks(epochs)
    ax.set_xlabel('epoch')
    ax.legend()
plt.show()

val_loss, val_acc = model.evaluate(val_data, verbose=0)
print(f'Final validation accuracy: {val_acc:.1%}')

## 8. Save the model

In [ ]:
MODEL_PATH       = '/content/plant_disease_model_v2.keras'
CLASS_NAMES_PATH = '/content/class_names_v2.json'
model.save(MODEL_PATH)
with open(CLASS_NAMES_PATH, 'w') as f:
    json.dump(class_names, f, indent=2)
print('Saved', MODEL_PATH, f'({os.path.getsize(MODEL_PATH) / 1e6:.1f} MB) and', CLASS_NAMES_PATH)

## 9. Test on the held-out photos

In [ ]:
test_data = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, class_names=class_names, shuffle=False, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
)
test_loss, test_acc = model.evaluate(test_data, verbose=0)
print(f'Test accuracy on {len(test_data.file_paths)} held-out photos: {test_acc:.1%}  (loss {test_loss:.3f})')

probs  = model.predict(test_data, verbose=0)
y_pred = probs.argmax(axis=1)
y_true = np.concatenate([y.numpy() for _, y in test_data])

results = []
for path, t, p, pr in zip(test_data.file_paths, y_true, y_pred, probs):
    results.append({
        'file': os.path.relpath(path, TEST_DIR),
        'true_label': class_names[t], 'pred_label': class_names[p],
        'confidence': round(float(pr[p]) * 100, 2), 'correct': bool(t == p),
    })

# Accuracy per crop
crop_of = lambda name: name.split('___')[0]
print(f"\n{'Crop':<30} Correct")
for crop in sorted({crop_of(c) for c in class_names}):
    rs = [r for r in results if crop_of(r['true_label']) == crop]
    print(f"{crop:<30} {sum(r['correct'] for r in rs)}/{len(rs)}")

wrong = [r for r in results if not r['correct']]
print(f'\nMisclassified ({len(wrong)}):')
for r in wrong:
    print(f"  {r['file']}\n    true: {r['true_label']}   predicted: {r['pred_label']} ({r['confidence']:.1f}%)")

report = {'total': len(results), 'correct': len(results) - len(wrong),
          'accuracy_pct': round(test_acc * 100, 2), 'per_image': results}
with open(TEST_DIR / 'test_report.json', 'w') as f:
    json.dump(report, f, indent=2)

In [ ]:
# Visual check of 40 random test photos: green = correct, red = wrong
sample = random.Random(0).sample(results, 40)
plt.figure(figsize=(20, 10))
for i, r in enumerate(sample):
    ax = plt.subplot(4, 10, i + 1)
    ax.imshow(tf.keras.utils.load_img(TEST_DIR / r['file'], target_size=IMG_SIZE)); ax.axis('off')
    ax.set_title(f"{r['pred_label'].replace('___', ' ')[:20]}\n{r['confidence']:.0f}%", fontsize=7,
                 color='green' if r['correct'] else 'red')
plt.suptitle(f'Held-out test photos (40 of {len(results)}): accuracy {test_acc:.1%}')
plt.tight_layout(); plt.show()

## 10. Download the test photos and the report

In [ ]:
shutil.make_archive('/content/test_samples_v2', 'zip', TEST_DIR)
files.download('/content/test_samples_v2.zip')